<a href="https://colab.research.google.com/github/anyahuang66/ds2002-fa26/blob/main/2026_09_18_pandas_challenge_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [ ]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units:,}")
print(f"There are {len(df)} orders in the dataset.")

Total revenue: $8,520.00
Total units: 783
There are 400 orders in the dataset.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [ ]:
# TODO
by_category = (
    df.groupby('category', as_index=False).agg(
        revenue=('revenue', 'sum'),
        units=('qty','sum')
    )
    .sort_values('revenue', ascending=False)
)

by_category['share_pct'] = (
    by_category['revenue']/df['revenue'].sum()*100
)

print(by_category.to_string(index=False, formatters={
    'revenue': '$(:,.2f)'.format,
    'share_pct': '{:.1f}%'.format
}))

category  revenue  units share_pct
    Food $(:,.2f)    362     50.4%
   Merch $(:,.2f)    158     20.8%
   Drink $(:,.2f)    178     18.2%
RainGear $(:,.2f)     85     10.6%


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [ ]:
# TODO
by_vendor = (
    df.groupby('vendor_id').agg(
        avg_revenue=('revenue', 'mean'),
        order_count=('revenue', 'size')
    )
    .sort_values('avg_revenue', ascending=False)
)

print(by_vendor.to_string(formatters={
    'avg_revenue': '${:,.2f}'.format
}))

top_vendor = by_vendor.index[0]
top_average = by_vendor.iloc[0]['avg_revenue']
top_count = int(by_vendor.iloc[0]['order_count'])

print(
    f"{top_vendor} has the highest average order revenue at "
    f"${top_average:,.2f}, based on {top_count} orders."
)

          avg_revenue  order_count
vendor_id                         
V-01           $22.60           94
V-18           $21.75          108
V-05           $20.58           93
V-10           $20.31          105
V-01 has the highest average order revenue at $22.60, based on 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [ ]:
# TODO
merch_revenue = df.loc[
    df['category'] == 'Merch',
    'revenue'
].sum()

merch_share = merch_revenue / df['revenue'].sum() *100

print(f"Merch accounts for {merch_share:.1f}% of total revenue.")

Merch accounts for 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [ ]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print("Rows before merge:", len(df))
print("Rows after merge:", len(joined))
print("Revenue before merge:", df['revenue'].sum())
print("Revenue after merge:", joined['revenue'].sum())


unmatched = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')

print("Unmatched vendor ID(s):", unmatched)

Rows before merge: 400
Rows after merge: 400
Revenue before merge: 8520.0
Revenue after merge: 8520.0
Unmatched vendor ID(s): ['V-18']


**The unmatched vendor, and what I did about it:** The unmatched vendor is V-18. I kept its orders and labeled the missing vendor name as "Unknown vendor." This keeps all 400 orders and the full $8,520.00 revenue total instead of losing those orders during the merge.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
# TODO
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(pivot.to_string(float_format=lambda x: f"${x:,.2f}"))

category            Drink      Food     Merch  RainGear     Total
vendor_name                                                      
Cav Merch North   $502.50 $1,054.50   $400.50   $175.50 $2,133.00
Hoos Burgers      $171.00 $1,338.00   $373.50   $241.50 $2,124.00
Rotunda Tacos     $298.50   $882.00   $489.00   $244.50 $1,914.00
Unknown vendor    $582.00 $1,018.50   $508.50   $240.00 $2,349.00
Total           $1,554.00 $4,293.00 $1,771.50   $901.50 $8,520.00


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell the vendors to pay more attention to Food, because it generated $4,293.00, or 50.4% of the $8,520.00 in total revenue. Merch generated $1,771.50, or 20.8% of total revenue, so it was also a significant part of sales. The vendors could use these numbers to make sure they have enough of the highest-revenue categories available while still keeping Merch stocked.

b) The least trustworthy answer is probably Q5's vendor-name report, because one vendor (V-18) is missing from the lookup table. V-18 represents 108 of the 400 orders, so it is not a very small group. I kept those orders as "unknown vendor," which preserves the revenue, but we still do not know the real vendor name. This means the vendor-level labels are incomplete even though the total revenue is correct.